# GiGPO Loss

**GiGPO: Group-in-Group Policy Optimization**

来自 arXiv:2504.02589，专为 **LLM Agent 训练**设计。

## 背景：GRPO 在 Agent 任务中的局限性

将 GRPO 直接用于 LLM Agent（多轮决策）时面临关键挑战：

**宏观信用分配问题**：
- GRPO 仅在序列级别分配奖励，整条轨迹内的所有 token 共享同一个优势值
- 无法区分轨迹中**哪些步骤**对成功/失败贡献更大
- 例如：Agent 在第 3 步选错了方向，但最终因运气因素成功。GRPO 会无差别地奖励全部步骤。

**稀疏奖励问题**：
- Agent 任务通常只在最后一步得到奖励（成功/失败）
- 中间步骤的优劣无法从单条轨迹中判断

**朴素解决方案的困境**：
- 对每个状态重新采样动作（per-state rollout）可以实现步骤级别信用分配
- 但代价极高：N 步轨迹需要额外 N 次推理，计算量翻 N 倍

## GiGPO 的核心思想：「群内群」

GiGPO 发现：当 N 条轨迹从**相同初始状态**出发，它们会**自然地经过许多相同的环境状态**。

这些「相同状态」就是天然的步骤级别比较机会！

通过「锚状态分组（Anchor State Grouping）」技术：
- 在已有的 N 条轨迹中，**离线**地按环境状态聚合动作
- 构建步骤级别分组，无需任何额外推理
- 计算代价可忽略不计（< 0.002% 额外时间）

## GiGPO 两层优势体系

### 第一层：Episode 级别优势 $A^E(\tau_i)$

与 GRPO 类似，比较整条轨迹的完整回报：
$$
A^E(\tau_i) = \frac{R(\tau_i) - \text{mean}\{R(\tau_j)\}_{j=1}^N}{F_{\text{norm}}\{R(\tau_j)\}_{j=1}^N}
$$

其中 $R(\tau_i) = \sum_t r_t^{(i)}$ 为轨迹 $\tau_i$ 的总回报。
$F_{\text{norm}}$ 可以是 std（标准 GRPO 方式）或常数 1（更稳定的 Leave-One-Out 方式）。

### 第二层：Step 级别优势 $A^S(a_t^{(i)})$

**锚状态分组**：令 $\mathcal{U} = \{\tilde{s}_1, \tilde{s}_2, \ldots, \tilde{s}_U\}$ 为所有轨迹中出现的唯一状态集合。

对每个锚状态 $\tilde{s} \in \mathcal{U}$，构造步骤级别分组：
$$
\mathcal{G}^S(\tilde{s}) = \left\{\left(a_t^{(i)}, R_t^{(i)}\right) \mid s_t^{(i)} = \tilde{s},\ 1 \leq i \leq N,\ 1 \leq t \leq T\right\}
$$

其中折扣回报 $R_t^{(i)} = \sum_{k=t}^T \gamma^{k-t} r_k^{(i)}$（捕捉当前动作的长期影响）。

然后计算步骤级别相对优势：
$$
A^S(a_t^{(i)}) = \frac{R_t^{(i)} - \text{mean}\left\{R_t^{(j)} \mid (a_t^{(j)}, R_t^{(j)}) \in \mathcal{G}^S(\tilde{s})\right\}}{F_{\text{norm}}\left\{R_t^{(j)} \mid (a_t^{(j)}, R_t^{(j)}) \in \mathcal{G}^S(\tilde{s})\right\}}
$$

### 组合优势

$$
\boxed{A(a_t^{(i)}) = A^E(\tau_i) + \omega \cdot A^S(a_t^{(i)})}
$$

其中 $\omega \geq 0$ 为平衡系数，控制步骤级别信号的权重。

## GiGPO 目标函数

$$
\mathcal{J}_{\text{GiGPO}}(\theta) = \mathbb{E}_{x\sim p(\mathcal{X}),\{\tau_i\}_{i=1}^N\sim\pi_{\theta_{\text{old}}}}
\frac{1}{NT}\sum_{i=1}^N\sum_{t=1}^T
\left[\min\left(\rho_\theta(a_t^{(i)}) A(a_t^{(i)}),\ \text{clip}(\rho_\theta(a_t^{(i)}), 1\pm\varepsilon) A(a_t^{(i)})\right) - \beta D_{\text{KL}}\right]
$$

其中 $\rho_\theta(a_t^{(i)}) = \dfrac{\pi_\theta(a_t^{(i)}|s_t^{(i)},x)}{\pi_{\theta_{\text{old}}}(a_t^{(i)}|s_t^{(i)},x)}$

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

## 数据结构：轨迹表示

In [ ]:
@dataclass
class Step:
    """
    Agent 轨迹中的单个步骤
    
    在 LLM Agent 场景中：
    - state: 环境状态（如 HTML 页面内容、任务描述等）的哈希表示
    - action: Agent 采取的动作（如点击、搜索等）的 log prob
    - reward: 该步骤的即时奖励
    """
    state: str          # 环境状态（用字符串表示，便于哈希比较）
    action_logprob: float       # 当前策略对该动作的 log prob
    old_action_logprob: float   # 旧策略对该动作的 log prob
    reward: float               # 该步骤的即时奖励


@dataclass
class Trajectory:
    """
    完整的 Agent 轨迹
    
    由多个步骤组成，最终得到一个完整回报。
    """
    steps: List[Step]
    
    @property
    def total_return(self):
        """轨迹总回报（所有步骤奖励之和）"""
        return sum(s.reward for s in self.steps)
    
    def discounted_returns(self, gamma=1.0):
        """
        计算每个步骤的折扣回报
        R_t = Σ_{k=t}^T γ^(k-t) * r_k
        
        这比即时奖励更好地衡量了动作的长期影响。
        当 γ=1 时退化为后缀奖励之和（episode to go）。
        
        Args:
            gamma: 折扣因子，[0, 1]
        Returns:
            list: 每个步骤的折扣回报
        """
        T = len(self.steps)
        disc_returns = [0.0] * T
        cumulative = 0.0
        for t in reversed(range(T)):
            cumulative = self.steps[t].reward + gamma * cumulative
            disc_returns[t] = cumulative
        return disc_returns

## Episode 级别优势 $A^E(\tau_i)$

In [ ]:
def compute_episode_advantage(trajectories: List[Trajectory], fnorm='std'):
    """
    计算 Episode 级别相对优势 A_E(τi)
    
    类似于 GRPO 在序列级别的操作：
    A_E(τi) = (R(τi) - mean(R)) / Fnorm(R)
    
    两种归一化方式：
    - 'std': 标准差归一化（与 GRPO 相同），在回报方差小时梯度可能过大
    - 'ones': F_norm=1，Leave-One-Out 估计量，更稳定（推荐用于 Agent 任务）
    
    Args:
        trajectories: N 条轨迹
        fnorm: 归一化方式，'std' 或 'ones'
    Returns:
        episode_advantages: 每条轨迹的 episode 级别优势，Tensor[N]
    """
    returns = torch.tensor([t.total_return for t in trajectories], dtype=torch.float32)
    mean_r = returns.mean()
    
    if fnorm == 'std':
        # 标准差归一化（与 GRPO 的 advantage 计算相同）
        norm_factor = returns.std() + 1e-5
    elif fnorm == 'ones':
        # F_norm = 1：Leave-One-Out 估计量，不受组内方差的影响
        # 适合 Agent 任务中简单任务/困难任务梯度不均匀的问题
        norm_factor = 1.0
    else:
        raise ValueError(f'Unknown fnorm: {fnorm}')
    
    episode_advantages = (returns - mean_r) / norm_factor  # [N]
    return episode_advantages

## 锚状态分组（Anchor State Grouping）

In [ ]:
def anchor_state_grouping(trajectories: List[Trajectory], gamma=1.0):
    """
    锚状态分组：从已有轨迹中离线构建步骤级别分组
    
    核心思想：
    - N 条轨迹从相同初始状态出发
    - 它们自然地会经过许多相同的环境状态
    - 对于每个唯一状态，聚合在该状态下采取的所有动作及其折扣回报
    - 这些组就是步骤级别比较的基础，无需额外推理
    
    与朴素 per-state rollout 的对比：
    - per-state rollout: 对每个状态重新采样 K 个动作，代价 = N * T * K 次推理
    - GiGPO: 离线操作，代价 = 哈希映射（O(NT) 时间，0 次额外推理）
    
    Args:
        trajectories: N 条轨迹
        gamma: 折扣因子
    Returns:
        state_groups: Dict[state -> List[(traj_idx, step_idx, disc_return)]]
                      每个唯一状态下的所有动作及其折扣回报
    """
    # state -> [(轨迹编号 i, 步骤编号 t, 折扣回报 R_t)]
    state_groups = defaultdict(list)
    
    for i, traj in enumerate(trajectories):
        # 计算该轨迹每个步骤的折扣回报
        disc_returns = traj.discounted_returns(gamma=gamma)
        
        for t, (step, disc_ret) in enumerate(zip(traj.steps, disc_returns)):
            # 将该步骤加入对应状态的分组
            state_groups[step.state].append((i, t, disc_ret))
    
    # 只保留有多个样本的状态（只有一个样本无法计算相对优势）
    state_groups = {s: group for s, group in state_groups.items() if len(group) >= 2}
    
    return state_groups

## Step 级别优势 $A^S(a_t^{(i)})$

In [ ]:
def compute_step_advantage(
    trajectories: List[Trajectory],
    gamma: float = 1.0,
    fnorm: str = 'std'
):
    """
    计算步骤级别相对优势 A_S(a_t^(i))
    
    对每个锚状态 s̃，在其分组 G_S(s̃) 内计算相对优势：
    A_S(a_t^(i)) = (R_t^(i) - mean_{G_S(s̃)}(R)) / Fnorm_{G_S(s̃)}(R)
    
    Args:
        trajectories: N 条轨迹
        gamma: 折扣因子
        fnorm: 归一化方式
    Returns:
        step_advantages: Dict[(i, t) -> float]，每个步骤的步骤级别优势
                         未出现在任何有效分组中的步骤，优势为 0
    """
    # 离线构建锚状态分组
    state_groups = anchor_state_grouping(trajectories, gamma=gamma)
    
    # (轨迹编号, 步骤编号) -> 步骤级别优势
    step_advantages = {}  # 默认 0（不在任何分组中）
    
    for state, group in state_groups.items():
        # 该锚状态下各动作的折扣回报
        disc_returns = torch.tensor([disc_ret for (_, _, disc_ret) in group])
        mean_r = disc_returns.mean()
        
        if fnorm == 'std':
            norm_factor = disc_returns.std() + 1e-5
        elif fnorm == 'ones':
            norm_factor = 1.0
        
        # 计算该分组内每个步骤的步骤级别优势
        for k, (i, t, disc_ret) in enumerate(group):
            adv = (disc_returns[k] - mean_r) / norm_factor
            step_advantages[(i, t)] = adv.item()
    
    return step_advantages

## 组合优势与 GiGPO Loss

In [ ]:
def gigpo_loss(
    trajectories: List[Trajectory],
    omega: float = 1.0,    # 步骤级别优势权重（平衡 episode 和 step 级别信号）
    epsilon: float = 0.2,  # IS 比率裁剪阈值
    gamma: float = 1.0,    # 折扣因子
    fnorm: str = 'std',    # 优势归一化方式
    is_debug: bool = True
):
    """
    GiGPO Loss 实现
    
    核心公式：
    J = (1/NT) * Σ_i Σ_t [min(ρ_t * A_combined, clip(ρ_t) * A_combined)] - β*KL
    
    其中：
    A_combined(a_t^i) = A_E(τi) + ω * A_S(a_t^i)
    - A_E: episode 级别优势（宏观信号，整条轨迹的好坏）
    - A_S: step 级别优势（微观信号，特定步骤相对其他轨迹同状态步骤的优劣）
    - ω: 两者之间的权重系数
    
    Args:
        trajectories: N 条轨迹（每条轨迹包含 T 个步骤）
        omega: step 优势的权重系数（ω=0 退化为普通 GRPO）
        epsilon: PPO 裁剪阈值
        gamma: 折扣因子
        fnorm: 优势归一化方式
    """
    N = len(trajectories)
    total_steps = sum(len(t.steps) for t in trajectories)
    
    # ============================================================
    # Step 1: 计算 Episode 级别优势 A_E(τi)
    # 从轨迹总回报出发，衡量每条轨迹的整体质量
    # ============================================================
    episode_advantages = compute_episode_advantage(trajectories, fnorm=fnorm)  # [N]
    
    # ============================================================
    # Step 2: 锚状态分组 + 计算 Step 级别优势 A_S(a_t^i)
    # 从相同状态下的不同动作出发，衡量每个步骤相对于同状态其他步骤的质量
    # 关键：这是离线操作，不需要额外推理！
    # ============================================================
    step_advantages = compute_step_advantage(trajectories, gamma=gamma, fnorm=fnorm)
    
    # ============================================================
    # Step 3: 计算每个步骤的组合优势
    # A_combined = A_E + ω * A_S
    # A_E: 宏观信号，鼓励整体更好的轨迹
    # A_S: 微观信号，精细调整特定状态下的动作选择
    # ============================================================
    total_loss = 0.0
    debug_info = []
    
    for i, traj in enumerate(trajectories):
        ae = episode_advantages[i].item()
        
        for t, step in enumerate(traj.steps):
            # 步骤级别优势（如果该步骤处于某个锚状态分组中）
            a_step = step_advantages.get((i, t), 0.0)  # 不在有效分组中则为 0
            
            # 组合优势：episode 宏观信号 + ω * step 微观信号
            a_combined = ae + omega * a_step
            
            # ============================================================
            # Step 4: PPO-style 损失（与 GRPO/DAPO 相同的更新目标）
            # 差异只在于优势函数的计算方式（两层 vs 单层）
            # ============================================================
            log_pi = torch.tensor(step.action_logprob)
            log_pi_old = torch.tensor(step.old_action_logprob)
            
            # IS 比率
            ratio = torch.exp(log_pi - log_pi_old)
            ratio_clip = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
            
            a_t = torch.tensor(a_combined)
            
            # min(r*A, clip(r)*A)
            pg = torch.minimum(ratio * a_t, ratio_clip * a_t)
            total_loss += pg
            
            if is_debug:
                debug_info.append({
                    'traj': i, 'step': t,
                    'state': step.state,
                    'reward': step.reward,
                    'A_E': ae,
                    'A_S': a_step,
                    'A_combined': a_combined,
                    'ratio': ratio.item()
                })
    
    # 取平均并加负号（loss = -J）
    loss = -(total_loss / total_steps)
    
    if is_debug:
        print(f'轨迹数 N={N}, 总步骤数 T_total={total_steps}')
        print(f'Episode advantages: {episode_advantages.tolist()}')
        print(f'锚状态分组数（有效步骤级别分组）: {len(step_advantages)}')
        print()
        print(f'{"轨迹":>4} {"步骤":>4} {"状态":>12} {"奖励":>6} {"A_E":>8} {"A_S":>8} {"A_组合":>8} {"IS比率":>8}')
        print('-' * 70)
        for info in debug_info:
            print(f'{info["traj"]:>4} {info["step"]:>4} {info["state"]:>12} '
                  f'{info["reward"]:>6.2f} {info["A_E"]:>8.3f} '
                  f'{info["A_S"]:>8.3f} {info["A_combined"]:>8.3f} {info["ratio"]:>8.3f}')
        print(f'\n[Loss]: {loss.item():.6f}')
    
    return loss

## 模拟 Agent 场景：WebShop 购物任务

In [ ]:
# 模拟 WebShop 场景：Agent 在电商网站上搜索并购买目标商品
#
# 场景描述：
# - 任务：搜索并购买「红色运动鞋，40码」
# - 3 条轨迹，每条最多 4 步
# - 步骤：搜索 → 浏览结果 → 查看详情 → 购买
#
# 关键：轨迹会经过相同的状态（如相同的搜索结果页面）

# 注：这里用字符串模拟环境状态，实际应用中是环境的哈希值

torch.manual_seed(42)

def rand_logprob(n=1):
    """生成随机 log 概率"""
    return float(torch.randn(n).clamp(-3, -0.1).item())

# 轨迹 1：搜索「红色运动鞋」→ 浏览第 1 个结果（不符合）→ 返回 → 购买第 2 个结果（成功）
traj1 = Trajectory(steps=[
    Step(state='homepage',         action_logprob=-0.8,  old_action_logprob=-1.0,  reward=0.0),
    Step(state='search_result_A',  action_logprob=-0.5,  old_action_logprob=-0.7,  reward=0.0),
    Step(state='product_page_1',   action_logprob=-0.9,  old_action_logprob=-1.2,  reward=0.0),
    Step(state='search_result_A',  action_logprob=-0.6,  old_action_logprob=-0.8,  reward=1.0),  # 成功购买
])

# 轨迹 2：搜索「运动鞋」（关键词不准确）→ 浏览结果 → 失败
traj2 = Trajectory(steps=[
    Step(state='homepage',         action_logprob=-1.2,  old_action_logprob=-1.0,  reward=0.0),
    Step(state='search_result_B',  action_logprob=-0.7,  old_action_logprob=-0.9,  reward=0.0),
    Step(state='product_page_3',   action_logprob=-0.8,  old_action_logprob=-1.1,  reward=0.0),
    Step(state='product_page_3',   action_logprob=-1.1,  old_action_logprob=-0.8,  reward=0.0),  # 未能购买
])

# 轨迹 3：搜索「红色运动鞋」→ 直接找到目标商品 → 成功（效率最高）
traj3 = Trajectory(steps=[
    Step(state='homepage',         action_logprob=-0.6,  old_action_logprob=-1.0,  reward=0.0),
    Step(state='search_result_A',  action_logprob=-0.4,  old_action_logprob=-0.7,  reward=0.0),  # 同状态！
    Step(state='product_page_2',   action_logprob=-0.5,  old_action_logprob=-0.6,  reward=1.0),  # 直接成功
    Step(state='checkout',         action_logprob=-0.7,  old_action_logprob=-0.9,  reward=0.0),
])

trajectories = [traj1, traj2, traj3]

# 打印轨迹信息
for i, traj in enumerate(trajectories):
    print(f'轨迹 {i+1}: 总回报={traj.total_return:.1f}, 步骤数={len(traj.steps)}')
    disc_ret = traj.discounted_returns(gamma=0.9)
    for t, (step, dr) in enumerate(zip(traj.steps, disc_ret)):
        print(f'  步骤 {t}: state={step.state:<20} reward={step.reward:.1f}  disc_return={dr:.3f}')
    print()

print()
print('注意：traj1 和 traj3 都经过了 search_result_A 状态，这是「锚状态」的来源！')

In [ ]:
# 演示锚状态分组
print('='*60)
print('锚状态分组结果（Anchor State Grouping）')
print('='*60)

state_groups = anchor_state_grouping(trajectories, gamma=0.9)

for state, group in state_groups.items():
    print(f'\n锚状态: "{state}" → {len(group)} 个动作进行比较')
    disc_returns = [dr for (_, _, dr) in group]
    for (i, t, dr) in group:
        print(f'  轨迹{i+1} 步骤{t}: 折扣回报={dr:.4f}')
    if len(disc_returns) > 1:
        mean_dr = sum(disc_returns) / len(disc_returns)
        std_dr = torch.tensor(disc_returns).std().item()
        print(f'  → 组内均值={mean_dr:.4f}, 标准差={std_dr:.4f}')
        print(f'  → 可以估计步骤级别优势了！')

print()
print('没有出现在分组中的状态（只有一条轨迹经过）：步骤优势默认为 0')

In [ ]:
# 计算完整 GiGPO Loss
print('='*60)
print('GiGPO Loss 计算（ω=1.0，sqrt 归一化）')
print('='*60)
loss = gigpo_loss(trajectories, omega=1.0, epsilon=0.2, gamma=0.9, fnorm='std')

## 消融实验：ω 对 Loss 的影响

In [ ]:
# 消融 ω：比较不同步骤级别权重的效果
omega_values = [0.0, 0.5, 1.0, 2.0, 5.0]
loss_values = []

print(f"{'omega':>8} | {'Loss':>12} | 说明")
print('-' * 50)

for omega in omega_values:
    loss = gigpo_loss(trajectories, omega=omega, is_debug=False)
    loss_values.append(loss.item())
    note = '（等价于 GRPO，仅 episode 级优势）' if omega == 0 else ''
    print(f'{omega:>8.1f} | {loss.item():>12.6f} | {note}')

plt.figure(figsize=(8, 4))
plt.plot(omega_values, loss_values, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=0, color='r', linestyle='--', alpha=0.5, label='ω=0: 等价于 GRPO')
plt.xlabel('ω（步骤级别优势权重）')
plt.ylabel('GiGPO Loss')
plt.title('ω 对 GiGPO Loss 的影响')
plt.legend()
plt.grid()
plt.show()

## 简化版 GiGPO Loss（Tensor 形式，适合 batch 训练）

In [ ]:
def gigpo_loss_tensor(
    pi_logprob,       # 当前策略 log prob，[N, T]（N 条轨迹，T 步）
    pi_old_logprob,   # 旧策略 log prob，[N, T]
    episode_adv,      # Episode 优势，[N]（每条轨迹一个值）
    step_adv,         # Step 优势，[N, T]（每个步骤一个值，不在分组中的为 0）
    mask,             # 有效步骤 mask，[N, T]
    omega=1.0,        # 步骤优势权重
    epsilon=0.2       # 裁剪阈值
):
    """
    GiGPO Loss 的 Tensor 批次实现
    
    适合实际训练中的批次操作，与 GRPO Loss 结构相似，
    唯一区别是优势函数从单层（episode）变为双层（episode + step）。
    
    Args:
        pi_logprob: [N, T]
        pi_old_logprob: [N, T]
        episode_adv: [N] episode 级别优势
        step_adv: [N, T] step 级别优势
        mask: [N, T] 有效步骤 mask（1=有效）
        omega: step 优势权重
        epsilon: PPO 裁剪阈值
    """
    # 组合优势：[N, 1] 广播 + ω * [N, T]
    A_combined = episode_adv.unsqueeze(1) + omega * step_adv  # [N, T]
    
    # IS 比率
    ratio = torch.exp(pi_logprob - pi_old_logprob)  # [N, T]
    ratio_clip = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
    
    # PPO min 操作
    pg = torch.minimum(ratio * A_combined, ratio_clip * A_combined)  # [N, T]
    
    # 用 mask 屏蔽无效步骤，用有效步骤总数归一化
    total_valid = mask.sum()
    loss = -(1.0 / total_valid) * (pg * mask).sum()
    
    return loss


# ======================== 测试 ========================
torch.manual_seed(0)
N, T = 3, 4  # 3 条轨迹，每条 4 步

pi_logprob_t = torch.randn(N, T) * 0.3 - 0.5
pi_old_logprob_t = torch.randn(N, T) * 0.3 - 0.5

# 模拟优势（实际应由 compute_episode_advantage 和 compute_step_advantage 计算）
episode_adv_t = torch.tensor([0.8, -1.2, 0.4])  # [N]
step_adv_t = torch.tensor([                       # [N, T]
    [0.5, 0.3, -0.2, 0.1],
    [-0.3, 0.0, 0.4, -0.5],
    [0.2, 0.6, 0.0, -0.1],
])
mask_t = torch.ones(N, T)  # 所有步骤都有效

loss_t = gigpo_loss_tensor(pi_logprob_t, pi_old_logprob_t,
                            episode_adv_t, step_adv_t, mask_t,
                            omega=1.0, epsilon=0.2)
print(f'GiGPO Loss (Tensor 实现): {loss_t.item():.6f}')

# ω=0 时等价于 GRPO
loss_grpo_equiv = gigpo_loss_tensor(pi_logprob_t, pi_old_logprob_t,
                                     episode_adv_t, torch.zeros_like(step_adv_t), mask_t,
                                     omega=0.0, epsilon=0.2)
print(f'GiGPO Loss (ω=0, 等价 GRPO): {loss_grpo_equiv.item():.6f}')

## 可视化：双层优势信号的作用

In [ ]:
# 可视化：展示两层优势信号的组合效果
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 假设 3 条轨迹，4 步，episode 优势已知
traj_labels = ['τ₁\n(成功)', 'τ₂\n(失败)', 'τ₃\n(成功)']
episode_advantages_vis = np.array([0.9, -1.3, 0.4])

# 步骤级别优势（只有经过共享状态的步骤才有非零值）
step_advantages_vis = np.array([
    [0.0, 0.8, 0.0, -0.3],   # τ1: 步骤1共享状态（搜索结果页）
    [0.0, 0.0, 0.0,  0.0],   # τ2: 未经过任何共享状态
    [0.0, -0.8, 0.0,  0.0],  # τ3: 步骤1共享状态（搜索后选择不同商品）
])

omega_vis = 1.0
combined_advantages_vis = episode_advantages_vis[:, None] + omega_vis * step_advantages_vis

step_labels = ['步骤0\n(搜索)', '步骤1\n(浏览)', '步骤2\n(查看)', '步骤3\n(购买)']
colors = ['steelblue', 'salmon', 'seagreen']
x = np.arange(4)

# 图1：Episode 优势
for i in range(3):
    axes[0].bar(x + i*0.25 - 0.25, [episode_advantages_vis[i]]*4,
                width=0.25, alpha=0.7, color=colors[i], label=traj_labels[i])
axes[0].set_xticks(x)
axes[0].set_xticklabels(step_labels)
axes[0].set_title('Episode 优势 A_E\n（所有步骤相同）')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].legend()
axes[0].grid(axis='y')

# 图2：Step 优势
for i in range(3):
    axes[1].bar(x + i*0.25 - 0.25, step_advantages_vis[i],
                width=0.25, alpha=0.7, color=colors[i], label=traj_labels[i])
axes[1].set_xticks(x)
axes[1].set_xticklabels(step_labels)
axes[1].set_title('Step 优势 ω*A_S\n（只有共享状态步骤有值）')
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].legend()
axes[1].grid(axis='y')

# 图3：组合优势
for i in range(3):
    axes[2].bar(x + i*0.25 - 0.25, combined_advantages_vis[i],
                width=0.25, alpha=0.7, color=colors[i], label=traj_labels[i])
axes[2].set_xticks(x)
axes[2].set_xticklabels(step_labels)
axes[2].set_title('组合优势 A_E + ω·A_S\n（宏微观信号结合）')
axes[2].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[2].legend()
axes[2].grid(axis='y')

plt.tight_layout()
plt.show()

print('结论：')
print('- Episode 优势（图1）：τ2 整体较差，所有步骤均被压制')
print('- Step 优势（图2）：τ1 在步骤1（浏览）上好于 τ3，但 τ3 在全局上同样成功')
print('- 组合优势（图3）：对 τ1 的步骤1给出更细粒度的正信号，对 τ3 的步骤1则更保守')
print('这就是 GiGPO 的核心优势：宏观引导（做对任务）+ 微观精调（做对每一步）')

## GiGPO vs GRPO 总结

| 特性 | GRPO | GiGPO |
|------|------|-------|
| 优势信号层次 | 单层（序列/轨迹级别）| 双层（episode + step 级别）|
| 信用分配粒度 | 整条轨迹共享同一优势 | 每个步骤有精细的信用 |
| 需要额外推理 | 否 | **否**（离线锚状态分组，0 额外推理）|
| 步骤级别信号来源 | 无 | 相同状态下不同轨迹的动作比较 |
| 适用场景 | 单轮 QA、推理任务 | 多轮 Agent、长时域决策任务 |
| ω=0 时 | N/A | 退化为标准 GRPO |
| 计算开销 | O(N) | O(NT)（哈希映射，极轻量）|

**核心直觉**：GiGPO 利用「相同任务下多条轨迹」的天然冗余，从已有数据中提取步骤级别的比较信息，
实现了「免费」的细粒度信用分配。这对 LLM Agent 训练至关重要，因为 Agent 的关键决策往往体现在少数几个转折步骤上。